# 🧪 Laboratorio 10: Interpretabilidad de Modelos con SHAP

**Módulo 03** | **Sesión 6** | **Duración estimada: 45min**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/laboratorios/lab_10_interpretabilidad_ejercicios.ipynb)

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Importancia nativa | Extraer e interpretar la importancia de variables de un Random Forest | Aplicación |
| 2 | Permutation importance | Comparar métodos de importancia y entender sus diferencias | Análisis |
| 3 | SHAP values | Calcular valores SHAP y crear summary plots | Aplicación |
| 4 | Importancia global SHAP | Interpretar la contribución media de cada variable | Análisis |
| 5 | Explicación individual | Explicar la predicción de un caso específico con waterfall plot | Análisis |
| 6 | Recomendación académica | Traducir resultados técnicos en recomendaciones de política universitaria | Evaluación |

## 💡 ¿Por qué es importante para tu práctica docente?

En el ámbito universitario, no basta con construir un modelo que prediga cuáles estudiantes están en riesgo académico. La pregunta clave es: **¿por qué están en riesgo?** Sin esta respuesta, no podemos diseñar intervenciones efectivas.

La **interpretabilidad de modelos** permite:

- **Explicar** a la coordinación académica qué factores contribuyen al riesgo estudiantil
- **Justificar** decisiones basadas en datos ante comités académicos
- **Identificar** palancas de intervención (ej. si la asistencia es el factor más importante, reforzar el seguimiento de asistencia)
- **Generar confianza** en los modelos predictivos entre directivos y profesores

En este laboratorio usarás **SHAP** (SHapley Additive exPlanations), el estándar actual para explicar modelos de machine learning, aplicado a datos de rendimiento académico de la facultad.

In [ ]:
# Librerías necesarias — ejecuta esta celda primero
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# SHAP para interpretabilidad
try:
    import shap
except ImportError:
    print('Instalando shap...')
    !pip install shap -q
    import shap

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')
print(f'SHAP versión: {shap.__version__}')

In [ ]:
# --- Preparación de datos (ejecuta esta celda sin modificar) ---

# Cargar dataset de rendimiento académico
df = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')
print(f'Dataset cargado: {df.shape[0]} estudiantes, {df.shape[1]} columnas')
print()

# Crear variable objetivo: en_riesgo = promedio < 10
df['en_riesgo'] = (df['promedio'] < 10).astype(int)
print(f'Estudiantes en riesgo (promedio < 10): {df["en_riesgo"].sum()} ({df["en_riesgo"].mean()*100:.1f}%)')
print(f'Estudiantes sin riesgo: {(1 - df["en_riesgo"]).sum()} ({(1 - df["en_riesgo"].mean())*100:.1f}%)')
print()

# Seleccionar features
features = ['semestre', 'edad', 'asistencia_pct', 'materias_inscritas', 'beca', 'trabaja']
df['beca'] = df['beca'].astype(int)
df['trabaja'] = df['trabaja'].astype(int)

X = df[features]
y = df['en_riesgo']

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} estudiantes')
print(f'Conjunto de prueba: {X_test.shape[0]} estudiantes')
print()
print('Features seleccionadas:')
print(X_train.head())

---

## ✏️ Ejercicio 1: Entrenar modelo y extraer importancia nativa

Entrena un Random Forest y extrae la importancia de variables:

1. Crea un `RandomForestClassifier` con `n_estimators=100` y `random_state=42`
2. Entrénalo con `X_train` y `y_train` usando `.fit()`
3. Extrae la importancia nativa con `.feature_importances_`
4. Crea un DataFrame con las columnas `'feature'` y `'importancia'`, ordenado de mayor a menor
5. Crea un **gráfico de barras horizontales** con la importancia de cada variable
6. **Comenta**: ¿cuál variable es la más importante según el modelo?

In [ ]:
# Ejercicio 1: Entrenar modelo y extraer importancia nativa
# Tu código aquí


---

## ✏️ Ejercicio 2: Permutation importance

Calcula la importancia por permutación y compárala con la nativa:

1. Usa `permutation_importance(modelo, X_test, y_test, n_repeats=10, random_state=42)`
2. Extrae `.importances_mean` del resultado
3. Crea un DataFrame con las columnas `'feature'` y `'importancia_permutacion'`, ordenado de mayor a menor
4. Imprime ambos rankings lado a lado (nativo vs. permutación)
5. **Comenta**: ¿coinciden los rankings? ¿Por qué podrían diferir?

**Pista**: la importancia nativa mide cuánto reduce cada variable la impureza (Gini) durante el entrenamiento, mientras que la permutación mide cuánto empeora el rendimiento al mezclar aleatoriamente los valores de cada variable en los datos de prueba.

In [ ]:
# Ejercicio 2: Permutation importance
# Tu código aquí


---

## ✏️ Ejercicio 3: SHAP TreeExplainer y summary plot

Calcula los valores SHAP y crea un summary plot:

1. Crea un explicador: `explainer = shap.TreeExplainer(modelo)`
2. Calcula los valores SHAP para el conjunto de prueba: `shap_values = explainer(X_test)`
3. Crea un summary plot con: `shap.summary_plot(shap_values[:, :, 1], X_test)`
   - Nota: usamos `[:, :, 1]` para seleccionar la clase 1 (en riesgo)
4. **Interpreta el gráfico**:
   - Cada punto es un estudiante
   - El eje X muestra el impacto SHAP (positivo = empuja hacia riesgo, negativo = empuja hacia seguridad)
   - El color indica el valor de la variable (rojo = alto, azul = bajo)
   - ¿Qué variables empujan más hacia el riesgo? ¿Cuáles protegen?

In [ ]:
# Ejercicio 3: SHAP TreeExplainer y summary plot
# Tu código aquí


---

## ✏️ Ejercicio 4: SHAP bar plot global

Crea un gráfico de barras con la importancia SHAP global:

1. Usa `shap.plots.bar(shap_values[:, :, 1])` para crear el bar plot
   - Este gráfico muestra el **valor absoluto medio** de los SHAP values por variable
2. Compara este ranking con el del Ejercicio 1 (importancia nativa del Random Forest)
3. **Escribe tu interpretación**: ¿coinciden los rankings? ¿Qué método te da más información y por qué?

**Pista**: SHAP no solo dice *cuánto* importa cada variable, sino también *en qué dirección* afecta la predicción.

In [ ]:
# Ejercicio 4: SHAP bar plot global
# Tu código aquí


---

## ✏️ Ejercicio 5: Waterfall plot para un caso individual

Explica la predicción de un estudiante específico en riesgo:

1. Identifica un estudiante en riesgo en `X_test`:
   ```python
   # Buscar índices de estudiantes en riesgo en el conjunto de prueba
   indices_riesgo = y_test[y_test == 1].index
   # Tomar la posición del primer estudiante en riesgo dentro de X_test
   pos = list(X_test.index).index(indices_riesgo[0])
   ```
2. Crea el waterfall plot: `shap.plots.waterfall(shap_values[pos, :, 1])`
3. Imprime los datos del estudiante seleccionado para contexto: `X_test.iloc[pos]`
4. **Explica**: ¿por qué el modelo clasificó a ESTE estudiante como en riesgo? ¿Cuáles factores contribuyeron más? ¿Alguno operó en contra del riesgo (valor SHAP negativo)?

In [ ]:
# Ejercicio 5: Waterfall plot para un caso individual
# Tu código aquí


---

## ✏️ Ejercicio 6: Recomendación para la coordinación académica

Traduce tus hallazgos técnicos en una recomendación accionable:

1. Basado en el análisis SHAP de los ejercicios anteriores, escribe una recomendación de **3 a 5 oraciones** dirigida a la coordinación académica de FACES
2. Tu recomendación debe responder:
   - ¿Cuáles son los **2-3 factores principales** que predicen el riesgo académico?
   - ¿Qué **indicadores** debería monitorear la coordinación para detección temprana?
   - ¿Qué **intervenciones concretas** tendrían mayor impacto según los datos?
3. Escribe tu recomendación en la celda de abajo (es una celda de código, usa `print()` o un comentario multilnea)

**Ejemplo de formato**:
```
RECOMENDACIÓN PARA LA COORDINACIÓN ACADÉMICA:

El análisis de interpretabilidad del modelo predictivo revela que...
Se recomienda monitorear especialmente...
Las intervenciones con mayor impacto potencial serían...
```

In [ ]:
# Ejercicio 6: Recomendación para la coordinación académica
# Tu código aquí


---

## 📋 Resumen

En este laboratorio practicaste:

| Paso | Qué hiciste |
|------|-------------|
| 1. Importancia nativa | Extrajiste la importancia de variables del Random Forest y la visualizaste |
| 2. Permutation importance | Comparaste dos métodos de importancia y entendiste sus diferencias |
| 3. SHAP summary plot | Visualizaste cómo cada variable afecta la predicción de riesgo |
| 4. SHAP bar plot | Comparaste la importancia global SHAP con la importancia nativa |
| 5. Waterfall plot | Explicaste por qué el modelo flagó a un estudiante específico como en riesgo |
| 6. Recomendación | Tradujiste resultados técnicos en recomendaciones de política académica |

## 📚 Referencias

1. Lundberg, S. M. & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions*. Advances in Neural Information Processing Systems 30 (NeurIPS). https://papers.nips.cc/paper/7062-a-unified-approach-to-interpreting-model-predictions

2. SHAP Documentation. *Welcome to the SHAP documentation*. https://shap.readthedocs.io/

3. Molnar, C. (2022). *Interpretable Machine Learning: A Guide for Making Black Box Models Explainable* (2nd ed.). https://christophm.github.io/interpretable-ml-book/

4. Scikit-learn developers. *Permutation feature importance*. https://scikit-learn.org/stable/modules/permutation_importance.html